# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tu-h-nguyn/FlyRank-End-to-End-Machine-Learning-Project-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)
Đây là bài toán Xếp hạng / Chấm điểm (Ranking / scoring). Chúng ta không phân loại đúng/sai đơn thuần, mà cần gán một trọng số ưu tiên (priority score) cho mỗi trang để trả lời câu hỏi "Nên xử lý trang nào trước?".

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Xác nhận loại bài toán và đầu ra dự kiến
ml_task_type = "Ranking / Scoring"
expected_output = "Ranked list of content_ids sorted by priority_score"

print(f"ML Task Type: {ml_task_type}")
print(f"Expected Output: {expected_output}")

ML Task Type: Ranking / Scoring
Expected Output: Ranked list of content_ids sorted by priority_score


## 2. Target or proxy

Mục tiêu lý tưởng (Target) phải là một kết quả đo lường được trong tương lai. Trên tập dữ liệu starter, chúng ta tạm dùng một biến đại diện (Proxy) dựa trên trạng thái suy giảm hiện tại (`trend_direction == 'down'`) để định hình logic, nhưng cần nhận thức rõ đây chỉ là proxy để tránh việc mô hình chỉ học lại một quy tắc có sẵn.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Định nghĩa logic của biến đại diện (Proxy)
proxy_target_name = "target_is_declining"
proxy_logic = "trend_direction == 'down'"

print(f"Proxy Target Column: {proxy_target_name}")
print(f"Logic used for Proxy: {proxy_logic}")
print("Lưu ý: Sẽ chuyển đổi giá trị logic này thành dạng số (1/0) trong dataframe.")

Proxy Target Column: target_is_declining
Logic used for Proxy: trend_direction == 'down'
Lưu ý: Sẽ chuyển đổi giá trị logic này thành dạng số (1/0) trong dataframe.


## 3. Success metric

Số liệu đánh giá là Precision@K (ví dụ: Precision@50). Hàm mất mát của bài toán không cần ưu tiên độ chính xác tuyệt đối trên toàn bộ tập dữ liệu, mà tập trung vào độ chuẩn xác của Top K trang đầu tiên mà đội ngũ có năng lực xử lý.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Khai báo tham số đánh giá
K = 50
success_metric = f"Precision@{K}"

print(f"Chỉ số đánh giá mục tiêu: {success_metric}")
print(f"Ý nghĩa nghiệp vụ: Trong {K} trang web được mô hình xếp hạng cao nhất, có bao nhiêu phần trăm thực sự là các trang đang suy giảm và cần ưu tiên xử lý.")

Chỉ số đánh giá mục tiêu: Precision@50
Ý nghĩa nghiệp vụ: Trong 50 trang web được mô hình xếp hạng cao nhất, có bao nhiêu phần trăm thực sự là các trang đang suy giảm và cần ưu tiên xử lý.


## 4. The unit of analysis, as a real dataframe

Đơn vị phân tích cốt lõi (grain) ở đây là một mục nội dung ẩn danh (`content_id`).

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import urllib.request
import pandas as pd
from typing import Optional

def load_and_frame_data() -> Optional[pd.DataFrame]:
    """Tải dữ liệu starter, xử lý đường dẫn động và định hình đơn vị phân tích (unit of analysis)."""
    # 1. Tự động xử lý đường dẫn tương thích mọi môi trường (Colab, Local)
    candidate_paths = [
        '../../data/raw/content_refresh_anonymized.csv',
        '../data/raw/content_refresh_anonymized.csv',
        'data/raw/content_refresh_anonymized.csv'
    ]
    filepath = next((p for p in candidate_paths if os.path.exists(p)), None)

    # Nếu không tìm thấy file cục bộ, tự động tải trực tiếp từ GitHub
    if not filepath:
        print("⚡ Không tìm thấy file cục bộ. Đang tải tập dữ liệu starter từ GitHub...")
        os.makedirs('data/raw', exist_ok=True)
        filepath = 'data/raw/content_refresh_anonymized.csv'
        raw_url = 'https://raw.githubusercontent.com/tu-h-nguyn/FlyRank-End-to-End-Machine-Learning-Project-Internship/main/data/raw/content_refresh_anonymized.csv'
        try:
            urllib.request.urlretrieve(raw_url, filepath)
            print("✅ Tải dữ liệu thành công!")
        except Exception as e:
            print(f"❌ Lỗi tải file: {e}")
            return None

    # 2. Xử lý dữ liệu
    try:
        df = pd.read_csv(filepath)

        # Lọc bỏ nhiễu: avg_position = 0 nghĩa là không có dữ liệu đo lường thực tế
        df_clean = df[df['avg_position'] > 0].copy()

        # Xây dựng cột Target (Proxy) từ dữ liệu quan sát được
        df_clean['target_is_declining'] = (df_clean['trend_direction'] == 'down').astype(int)

        # Chọn các cột cốt lõi để hiển thị grain (1 dòng = 1 content_id)
        cols_to_show = [
            'content_id', 'client_id', 'impressions_90d',
            'avg_position', 'ctr', 'target_is_declining'
        ]

        return df_clean[cols_to_show].head()
    except Exception as e:
        print(f"Lỗi xử lý dataframe: {e}")
        return None

# Chạy hàm và hiển thị kết quả (không cần truyền đường dẫn cứng nữa)
sample_df = load_and_frame_data()
display(sample_df)

⚡ Không tìm thấy file cục bộ. Đang tải tập dữ liệu starter từ GitHub...
✅ Tải dữ liệu thành công!


,content_id,client_id,impressions_90d,avg_position,ctr,target_is_declining
0,content_304f48230142,client_f369cb89fc,3803,10.6,0.76,1
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,0.05,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,0.09,1
3,content_331d6c4de07b,client_19581e27de,11751,6.2,0.49,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,0.13,1


Nếu đưa `trend_direction` hoặc `trend_pct` vào ma trận đặc trưng để dự đoán `target_is_declining`, hệ thống sẽ gặp phải lỗi rò rỉ dữ liệu (Data Leakage) dẫn đến kết quả vòng tròn (circular result). Vì `trend_direction` được tính trực tiếp từ `trend_pct`, và nhãn lại được tạo ra từ `trend_direction`, mô hình sẽ đạt độ chính xác 100% bằng cách học thuộc công thức này thay vì tự tìm ra quy luật từ các tín hiệu thật (như lượng hiển thị hay tỷ lệ click). Đó là lý do hai cột này tuyệt đối không bao giờ được dùng làm features.

## 5. Why ML beats a fixed rule here

Một hệ thống quy tắc (if-statement) tĩnh không thể nắm bắt sự tương tác phi tuyến tính giữa các chiều dữ liệu đa dạng như tỷ lệ cuộn (`scroll_rate`), tỷ lệ nhấp chuột `ctr` (được biểu diễn dưới dạng tỷ lệ nhân 100) hay vị trí trung bình. Học máy giúp tự động tìm ra trọng số tối ưu từ các cụm tín hiệu nhiễu này.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Liệt kê các chiều dữ liệu (features) cho thấy sự phức tạp phi tuyến tính
features_to_model = [
    "scroll_rate",
    "ctr (đã nhân 100)",
    "avg_position",
    "impressions_90d",
    "content_age_days"
]

print("Danh sách các tín hiệu phức tạp cần kết hợp:")
for feature in features_to_model:
    print(f" - {feature}")

print("\n=> Kết luận: Một hệ thống IF-ELSE thủ công sẽ không thể tìm ra ngưỡng (threshold) tối ưu đồng thời cho ngần này biến. ML sẽ làm tốt hơn.")

Danh sách các tín hiệu phức tạp cần kết hợp:
 - scroll_rate
 - ctr (đã nhân 100)
 - avg_position
 - impressions_90d
 - content_age_days

=> Kết luận: Một hệ thống IF-ELSE thủ công sẽ không thể tìm ra ngưỡng (threshold) tối ưu đồng thời cho ngần này biến. ML sẽ làm tốt hơn.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.